# 04 — Modeling & Evaluation

**Points-finish classification using integrated race-session features** (Tier 1 early-session + Tier 2 full-session analytical).

Season-based splits, baselines, supervised models, and model result artifacts.

- **Target:** `points_finish`
- **Default X:** 40 numeric features from `model_feature_plan.csv` (frozen)
- **Models:** Random, majority, heuristic baselines; Logistic Regression, Random Forest, LightGBM
- **Split:** Train 2023 · Validation 2024 · Test 2025 (season-based, no random row split)
- **Engine:** pandas + scikit-learn + LightGBM (Gold mart handoff)

Run after `03_gold_feature_engineering.ipynb` with the same `USE_GOOGLE_DRIVE` setting.


## Colab setup (required every session)

Identical to `00`–`03`: clone, `pip install -e .`, Drive mount, set `OPENF1_DATA_ROOT` **before** importing config.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Configuration


In [2]:
MODELING_MODE = "full"  # "smoke" for wiring-only checks; "full" for the official 2023-2025 season splits
CLEAR_MODEL_OUTPUTS = True

from pathlib import Path

import pandas as pd

from openf1_pipeline.config import (
    RANDOM_SEED,
    get_data_quality_reports_dir,
    get_feature_definitions_dir,
    get_gold_dir,
    get_manifests_dir,
    get_model_results_dir,
    get_output_root,
)
from openf1_pipeline.features.feature_dictionary import validate_no_leakage
from openf1_pipeline.gold.build_feature_mart import GOLD_MART_FILENAME, TARGET_COLUMN
from openf1_pipeline.modeling.baselines import (
    heuristic_position_baseline,
    majority_class_baseline,
    random_baseline_predictions,
)
from openf1_pipeline.modeling.feature_selection import (
    save_model_feature_plan,
    summarize_feature_tiers,
)
from openf1_pipeline.modeling.evaluate import (
    build_error_analysis,
    compute_classification_metrics,
    compute_confusion_matrix_table,
    save_modeling_outputs,
)
from openf1_pipeline.modeling.splits import resolve_modeling_splits
from openf1_pipeline.modeling.train import (
    build_lightgbm_pipeline,
    build_logistic_regression_pipeline,
    build_random_forest_pipeline,
    extract_feature_importance,
    get_model_feature_columns,
    prepare_model_matrix,
    train_models,
)
from openf1_pipeline.utils.cleanup import clean_model_outputs
from openf1_pipeline.utils.io import read_parquet_if_exists

GOLD_DIR = get_gold_dir()
FEATURE_DEFINITIONS_DIR = get_feature_definitions_dir()
MODEL_RESULTS_DIR = get_model_results_dir()
MANIFESTS_DIR = get_manifests_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()

MART_PATH = GOLD_DIR / GOLD_MART_FILENAME
DICT_PATH = FEATURE_DEFINITIONS_DIR / "feature_dictionary.csv"
PLAN_PATH = FEATURE_DEFINITIONS_DIR / "model_feature_plan.csv"
LEAKAGE_PATH = DATA_QUALITY_REPORTS_DIR / "gold_leakage_guard_report.csv"

print("MODELING_MODE:", MODELING_MODE)
print("CLEAR_MODEL_OUTPUTS:", CLEAR_MODEL_OUTPUTS)
print("MART_PATH:", MART_PATH)
print("DICT_PATH:", DICT_PATH)
print("PLAN_PATH:", PLAN_PATH)


MODELING_MODE: full
CLEAR_MODEL_OUTPUTS: True
MART_PATH: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold/driver_race_feature_mart.parquet
DICT_PATH: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/feature_definitions/feature_dictionary.csv
PLAN_PATH: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/feature_definitions/model_feature_plan.csv


## Clean model outputs


In [3]:
if CLEAR_MODEL_OUTPUTS:
    print("Cleaning model results...")
    clean_model_outputs(model_results_dir=MODEL_RESULTS_DIR, manifests_dir=MANIFESTS_DIR)
else:
    print("Skipping model cleanup (CLEAR_MODEL_OUTPUTS=False).")


INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results (0 items removed)
INFO:openf1_pipeline.utils.cleanup:Model outputs cleanup complete: {'model_results': 0, 'model_run_manifest': 0}


Cleaning model results...


## Load Gold mart, feature dictionary, and model feature plan


In [4]:
if not MART_PATH.exists():
    raise FileNotFoundError(
        f"Gold mart missing: {MART_PATH}. Run 03_gold_feature_engineering.ipynb first."
    )
if not DICT_PATH.is_file():
    raise FileNotFoundError(
        f"Feature dictionary missing: {DICT_PATH}. Run notebook 03 first."
    )

gold_df = read_parquet_if_exists(MART_PATH)
if gold_df is None or gold_df.empty:
    raise ValueError(f"Gold mart at {MART_PATH} is missing or empty.")

feature_dict = pd.read_csv(DICT_PATH)
leakage_report = pd.read_csv(LEAKAGE_PATH) if LEAKAGE_PATH.is_file() else None

if not PLAN_PATH.is_file():
    saved = save_model_feature_plan(PLAN_PATH)
    print("Created frozen model feature plan:", saved)
else:
    print("Loaded existing model feature plan:", PLAN_PATH)

feature_plan = pd.read_csv(PLAN_PATH)
tier_counts = summarize_feature_tiers(feature_plan)
print("Gold shape:", gold_df.shape)
print("Target rate:", gold_df[TARGET_COLUMN].mean())
validate_no_leakage(gold_df, feature_dict)
print("Leakage guard: OK")
if leakage_report is not None:
    blocked = leakage_report.loc[leakage_report["allowed_for_modeling"] == False]
    print(f"Leakage report: {len(blocked)} columns blocked from modeling")

print("Feature plan tier counts (default_include=True):", tier_counts)
display(feature_plan.loc[feature_plan["default_include"] == True])

feature_columns = get_model_feature_columns(feature_dict)
print(f"Selected model features ({len(feature_columns)}):")
print(feature_columns)


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/feature_definitions/model_feature_plan.csv (62 rows)


Created frozen model feature plan: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/feature_definitions/model_feature_plan.csv
Gold shape: (1756, 62)
Target rate: 0.47494305239179957
Leakage guard: OK
Leakage report: 22 columns blocked from modeling
Feature plan tier counts (default_include=True): {'tier1_early': 8, 'tier2_full_session': 32, 'total_default_numeric': 40}


,feature_name,feature_tier,feature_group,default_include,allowed_for_modeling,reason
10,avg_first5_lap_duration,tier1_early,laps,True,True,Default numeric model feature (Tier 1 early-se...
11,best_first5_lap_duration,tier1_early,laps,True,True,Default numeric model feature (Tier 1 early-se...
12,std_first5_lap_duration,tier1_early,laps,True,True,Default numeric model feature (Tier 1 early-se...
13,first_observed_position,tier1_early,position,True,True,Default numeric model feature (Tier 1 early-se...
14,early_avg_position,tier1_early,position,True,True,Default numeric model feature (Tier 1 early-se...
15,early_min_position,tier1_early,position,True,True,Default numeric model feature (Tier 1 early-se...
16,early_max_position,tier1_early,position,True,True,Default numeric model feature (Tier 1 early-se...
17,position_observation_count,tier1_early,position,True,True,Default numeric model feature (Tier 1 early-se...
18,lap_count,tier2_full_session,laps,True,True,Default numeric model feature (Tier 2 full-ses...
19,avg_lap_duration,tier2_full_session,laps,True,True,Default numeric model feature (Tier 2 full-ses...


INFO:openf1_pipeline.modeling.train:Resolved 40 model feature columns


Selected model features (40):
['avg_first5_lap_duration', 'best_first5_lap_duration', 'std_first5_lap_duration', 'first_observed_position', 'early_avg_position', 'early_min_position', 'early_max_position', 'position_observation_count', 'lap_count', 'avg_lap_duration', 'median_lap_duration', 'best_lap_duration', 'std_lap_duration', 'avg_sector_1', 'avg_sector_2', 'avg_sector_3', 'avg_i1_speed', 'avg_i2_speed', 'avg_st_speed', 'pit_out_lap_count', 'pit_stop_count', 'avg_pit_duration', 'min_pit_duration', 'max_pit_duration', 'first_pit_lap', 'early_pit_stop_flag', 'avg_air_temperature', 'avg_track_temperature', 'avg_humidity', 'avg_pressure', 'avg_wind_speed', 'rainfall_mean', 'rainfall_flag', 'race_control_message_count', 'flag_message_count', 'yellow_flag_count', 'red_flag_count', 'green_flag_count', 'safety_car_message_count', 'pit_exit_message_count']


## Resolve train/validation/test splits


In [5]:
splits, split_meta = resolve_modeling_splits(gold_df, mode=MODELING_MODE)
train_df = splits["train"]
val_df = splits["validation"]
test_df = splits["test"]

print("Split metadata:", split_meta)
for name, part in splits.items():
    if part.empty:
        print(f"WARNING: split '{name}' is empty")
    else:
        rate = part[TARGET_COLUMN].mean()
        seasons = sorted(part["session_year"].dropna().unique().tolist()) if "session_year" in part.columns else []
        print(f"{name}: n={len(part)}, points_finish rate={rate:.4f}, seasons={seasons}")

if MODELING_MODE == "smoke":
    print("SMOKE MODE: wiring verification only — metrics are NOT official evidence.")
elif MODELING_MODE == "full":
    expected = {"train": {2023}, "validation": {2024}, "test": {2025}}
    for split_name, part in splits.items():
        if part.empty:
            raise ValueError(
                f"MODELING_MODE=full requires non-empty '{split_name}' split. "
                "Run full 2023–2025 Gold ingest first."
            )
        seasons = set(pd.to_numeric(part["session_year"], errors="coerce").dropna().astype(int))
        if seasons != expected[split_name]:
            raise ValueError(
                f"MODELING_MODE=full: split '{split_name}' seasons {seasons} != {expected[split_name]}"
            )
    print("FULL MODE: season splits 2023/2024/2025 verified — metrics may be used as official evidence.")


INFO:openf1_pipeline.modeling.splits:Split 'train': 558 rows
INFO:openf1_pipeline.modeling.splits:Split 'validation': 599 rows
INFO:openf1_pipeline.modeling.splits:Split 'test': 599 rows


Split metadata: {'split_method': 'season', 'evidence_tier': 'mba_official'}
train: n=558, points_finish rate=0.4624, seasons=[2023]
validation: n=599, points_finish rate=0.4808, seasons=[2024]
test: n=599, points_finish rate=0.4808, seasons=[2025]
FULL MODE: season splits 2023/2024/2025 verified — metrics may be used as official evidence.


## Save train/validation/test split summary

In [6]:
from openf1_pipeline.utils.io import save_dataframe_csv

split_summary_rows = []
for name, part in splits.items():
    if part.empty:
        split_summary_rows.append(
            {
                "split": name,
                "season": None,
                "n_rows": 0,
                "points_finish_pos": 0,
                "points_finish_neg": 0,
                "pos_rate": None,
            }
        )
        continue
    seasons_present = sorted(
        pd.to_numeric(part["session_year"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    season_value = (
        seasons_present[0]
        if len(seasons_present) == 1
        else ";".join(str(s) for s in seasons_present)
    )
    pos = int(part[TARGET_COLUMN].sum())
    neg = int((part[TARGET_COLUMN] == 0).sum())
    split_summary_rows.append(
        {
            "split": name,
            "season": season_value,
            "n_rows": int(len(part)),
            "points_finish_pos": pos,
            "points_finish_neg": neg,
            "pos_rate": round(float(part[TARGET_COLUMN].mean()), 4),
        }
    )

split_summary_df = pd.DataFrame(
    split_summary_rows,
    columns=[
        "split",
        "season",
        "n_rows",
        "points_finish_pos",
        "points_finish_neg",
        "pos_rate",
    ],
)
split_summary_path = MODEL_RESULTS_DIR / "train_validation_test_split_summary.csv"
save_dataframe_csv(split_summary_df, split_summary_path)
print("Saved split summary:", split_summary_path)
display(split_summary_df)

INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/train_validation_test_split_summary.csv (3 rows)


Saved split summary: /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/train_validation_test_split_summary.csv


,split,season,n_rows,points_finish_pos,points_finish_neg,pos_rate
0,train,2023,558,258,300,0.4624
1,validation,2024,599,288,311,0.4808
2,test,2025,599,288,311,0.4808


## Baselines


In [7]:
X_train, y_train = prepare_model_matrix(train_df, feature_columns)
X_val, y_val = prepare_model_matrix(val_df, feature_columns)
X_test, y_test = prepare_model_matrix(test_df, feature_columns)

baseline_rows = []
cm_parts = []
error_parts = []

for split_name, eval_df, y_true in [
    ("validation", val_df, y_val),
    ("test", test_df, y_test),
]:
    if len(eval_df) == 0:
        continue
    rand_pred, rand_proba = random_baseline_predictions(
        y_true, positive_rate=float(y_train.mean()) if len(y_train) else None
    )
    baseline_rows.append(
        compute_classification_metrics(y_true, rand_pred, rand_proba, "random_baseline", split_name)
    )
    cm_parts.append(compute_confusion_matrix_table(y_true, rand_pred, "random_baseline", split_name))
    error_parts.append(
        build_error_analysis(eval_df, y_true, rand_pred, rand_proba, "random_baseline", split_name)
    )

    maj_pred, maj_proba = majority_class_baseline(y_train, y_true)
    baseline_rows.append(
        compute_classification_metrics(y_true, maj_pred, maj_proba, "majority_baseline", split_name)
    )
    cm_parts.append(compute_confusion_matrix_table(y_true, maj_pred, "majority_baseline", split_name))
    error_parts.append(
        build_error_analysis(eval_df, y_true, maj_pred, maj_proba, "majority_baseline", split_name)
    )

    heur_pred, heur_proba = heuristic_position_baseline(eval_df)
    baseline_rows.append(
        compute_classification_metrics(y_true, heur_pred, heur_proba, "heuristic_position", split_name)
    )
    cm_parts.append(compute_confusion_matrix_table(y_true, heur_pred, "heuristic_position", split_name))
    error_parts.append(
        build_error_analysis(eval_df, y_true, heur_pred, heur_proba, "heuristic_position", split_name)
    )

baseline_metrics = pd.DataFrame(baseline_rows)
if MODELING_MODE == "smoke":
    print("SMOKE OUTPUT: baseline metrics below are wiring checks only.")
display(baseline_metrics)


/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = subset.groupby(col, dropna=False).size().reset_index(name="count")
/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = subset.groupby(col, dropna=False).size().reset_index(name="count")
/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=

,model_name,split,accuracy,precision,recall,f1,roc_auc,positive_rate_true,positive_rate_pred,n_rows
0,random_baseline,validation,0.4975,0.4765,0.4583,0.4673,0.5000,0.4808,0.4624,599
1,majority_baseline,validation,0.5192,0.0000,0.0000,0.0000,0.5000,0.4808,0.0000,599
2,heuristic_position,validation,0.8397,0.8200,0.8542,0.8367,0.8403,0.4808,0.5008,599
3,random_baseline,test,0.4975,0.4765,0.4583,0.4673,0.5000,0.4808,0.4624,599
4,majority_baseline,test,0.5192,0.0000,0.0000,0.0000,0.5000,0.4808,0.0000,599
5,heuristic_position,test,0.7796,0.7600,0.7917,0.7755,0.7801,0.4808,0.5008,599


## Train supervised models


In [8]:
model_specs = {
    "logistic_regression": build_logistic_regression_pipeline(feature_columns, X_train),
    "random_forest": build_random_forest_pipeline(feature_columns, X_train),
    "lightgbm": build_lightgbm_pipeline(feature_columns, X_train),
}
fitted_models = train_models(X_train, y_train, model_specs)
list(fitted_models.keys())


INFO:openf1_pipeline.modeling.train:Training model: logistic_regression (558 rows)
INFO:openf1_pipeline.modeling.train:Training model: random_forest (558 rows)
INFO:openf1_pipeline.modeling.train:Training model: lightgbm (558 rows)


['logistic_regression', 'random_forest', 'lightgbm']

## Validation and test evaluation


In [9]:
val_rows = []
test_rows = []
importance_parts = []

for model_name, pipeline in fitted_models.items():
    if len(X_val) > 0:
        val_pred = pipeline.predict(X_val)
        val_proba = pipeline.predict_proba(X_val)[:, 1] if hasattr(pipeline, "predict_proba") else None
        val_rows.append(compute_classification_metrics(y_val, val_pred, val_proba, model_name, "validation"))
        cm_parts.append(compute_confusion_matrix_table(y_val, val_pred, model_name, "validation"))
        error_parts.append(build_error_analysis(val_df, y_val, val_pred, val_proba, model_name, "validation"))
    if len(X_test) > 0:
        test_pred = pipeline.predict(X_test)
        test_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
        test_rows.append(compute_classification_metrics(y_test, test_pred, test_proba, model_name, "test"))
        cm_parts.append(compute_confusion_matrix_table(y_test, test_pred, model_name, "test"))
        error_parts.append(build_error_analysis(test_df, y_test, test_pred, test_proba, model_name, "test"))
    importance_parts.append(extract_feature_importance(model_name, pipeline, feature_columns))

validation_metrics = pd.DataFrame(val_rows)
test_metrics = pd.DataFrame(test_rows)
confusion_matrix_df = pd.concat(cm_parts, ignore_index=True) if cm_parts else pd.DataFrame()
error_analysis_df = pd.concat(error_parts, ignore_index=True) if error_parts else pd.DataFrame()
feature_importance_df = pd.concat(importance_parts, ignore_index=True) if importance_parts else pd.DataFrame()

display(validation_metrics)
display(test_metrics)


/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = subset.groupby(col, dropna=False).size().reset_index(name="count")
/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = subset.groupby(col, dropna=False).size().reset_index(name="count")
/content/openf1-big-data-pipeline/src/openf1_pipeline/modeling/evaluate.py:139: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=

,model_name,split,accuracy,precision,recall,f1,roc_auc,positive_rate_true,positive_rate_pred,n_rows
0,logistic_regression,validation,0.7763,0.8130,0.6944,0.7491,0.8443,0.4808,0.4107,599
1,random_forest,validation,0.8464,0.8525,0.8229,0.8375,0.9212,0.4808,0.4641,599
2,lightgbm,validation,0.8080,0.8216,0.7674,0.7935,0.8887,0.4808,0.4491,599


,model_name,split,accuracy,precision,recall,f1,roc_auc,positive_rate_true,positive_rate_pred,n_rows
0,logistic_regression,test,0.7329,0.6951,0.7917,0.7403,0.8088,0.4808,0.5476,599
1,random_forest,test,0.7963,0.8007,0.7674,0.7837,0.8733,0.4808,0.4608,599
2,lightgbm,test,0.7813,0.7794,0.7604,0.7698,0.8590,0.4808,0.4691,599


## Save modeling outputs


In [10]:
output_paths = save_modeling_outputs(
    baseline_metrics=baseline_metrics,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    confusion_matrix_df=confusion_matrix_df,
    error_analysis_df=error_analysis_df,
    feature_importance_df=feature_importance_df,
    model_results_dir=MODEL_RESULTS_DIR,
    manifests_dir=MANIFESTS_DIR,
    manifest_extra={
        "modeling_mode": MODELING_MODE,
        "split_method": split_meta.get("split_method"),
        "evidence_tier": split_meta.get("evidence_tier"),
        "random_seed": RANDOM_SEED,
        "target": TARGET_COLUMN,
        "feature_count": len(feature_columns),
        "feature_tier_counts": tier_counts,
        "models": list(fitted_models.keys()),
        "baselines": ["random_baseline", "majority_baseline", "heuristic_position"],
    },
)
output_paths


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/baseline_metrics.csv (6 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/validation_metrics.csv (3 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/test_metrics.csv (3 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/confusion_matrix.csv (48 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/error_analysis.csv (807 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/feature_importance.csv (120 rows)
INFO:openf1_pipeline.utils.io:Wrote manifest /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/model_run_manifest.json
INFO:openf1_pipeline

{'baseline_metrics': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/baseline_metrics.csv'),
 'validation_metrics': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/validation_metrics.csv'),
 'test_metrics': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/test_metrics.csv'),
 'confusion_matrix': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/confusion_matrix.csv'),
 'error_analysis': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/error_analysis.csv'),
 'feature_importance': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/model_results/feature_importance.csv'),
 'model_run_manifest': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/model_run_manifest.json')}

## Next steps

- [ ] Review `reports/model_results/` CSVs on Drive
- [ ] For official evidence: set `MODELING_MODE = "full"` after the 2023–2025 Gold run
- [ ] Proceed to **05 — report artifacts** to build tables and figures
